In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Home_work").getOrCreate()
print(spark.sparkContext.master)
print(spark.version)

local[*]
3.5.0


In [2]:
df = (
    spark.read
    .option('header', True)
    .option('sep',',')
    .option('inferSchema', True)
    .csv('../data/covid-data.csv')
)

### 1. 15 стран с наибольшим процентом переболевших на 31 марта: 

In [3]:
filtered_df = df.select('iso_code', 'location', 'total_cases', 'population') \
    .filter(
        col("total_cases").isNotNull() & 
        col("population").isNotNull() & 
        (col("population") > 0) &
        (col('date') == '2021-03-31') &
        ~col('iso_code').like('OWID%'))

filtered_df.withColumn('cases_percent', 
    round(col('total_cases') / col('population') * 100, 2)) \
    .select('iso_code', 'location', 'cases_percent').sort(col('cases_percent').desc()).show(15)


+--------+-------------+-------------+
|iso_code|     location|cases_percent|
+--------+-------------+-------------+
|     AND|      Andorra|        15.54|
|     MNE|   Montenegro|        14.52|
|     CZE|      Czechia|        14.31|
|     SMR|   San Marino|        13.94|
|     SVN|     Slovenia|        10.37|
|     LUX|   Luxembourg|         9.85|
|     ISR|       Israel|         9.63|
|     USA|United States|          9.2|
|     SRB|       Serbia|         8.83|
|     BHR|      Bahrain|         8.49|
|     PAN|       Panama|         8.23|
|     PRT|     Portugal|         8.06|
|     EST|      Estonia|         8.02|
|     SWE|       Sweden|         7.97|
|     LTU|    Lithuania|         7.94|
+--------+-------------+-------------+
only showing top 15 rows



### 2. Top 10 стран с максимальным зафиксированным кол-вом новых случаев за последнюю неделю марта 2021

In [4]:
filtered_df = df.select('date', 'location', 'new_cases') \
    .filter(
        (col('date') <= '2021-03-31') &
        (col('date') >= '2021-03-25') &
        col("new_cases").isNotNull() &
        ~col('iso_code').like('OWID%'))
    
window_spec = Window.partitionBy('location').orderBy(col('new_cases').desc(), col('date').desc())

filtered_df.withColumn('rn', row_number() \
    .over(window_spec)) \
    .filter(col("rn") == 1) \
    .drop("rn") \
    .orderBy(col('new_cases').desc()) \
    .show(10)  

+----------+-------------+---------+
|      date|     location|new_cases|
+----------+-------------+---------+
|2021-03-25|       Brazil| 100158.0|
|2021-03-26|United States|  77321.0|
|2021-03-31|        India|  72330.0|
|2021-03-31|       France|  59054.0|
|2021-03-31|       Turkey|  39302.0|
|2021-03-26|       Poland|  35145.0|
|2021-03-31|      Germany|  25014.0|
|2021-03-26|        Italy|  24076.0|
|2021-03-25|         Peru|  19206.0|
|2021-03-26|      Ukraine|  18226.0|
+----------+-------------+---------+
only showing top 10 rows



### 3. Подсчет изменения случаев относительно предыдущего дня в России за последнюю неделю марта 2021. 

In [5]:
filtered_df = df.select('date', 'new_cases') \
    .filter(
        (col('date') <= '2021-03-31') &
        (col('date') >= '2021-03-24') &
        col('location').like('Rus%') &
        ~col('iso_code').like('OWID%'))
window_spec = Window.orderBy('date')
filtered_df.withColumn('new_cases_yesterday', lag('new_cases').over(window_spec)) \
    .withColumn('delta', col('new_cases') - col('new_cases_yesterday')) \
    .select(col('date'), 
            col('new_cases_yesterday'), 
            col('new_cases').alias('new_cases_today'),
            col('delta')) \
    .where(col('date') > '2021-03-24') \
    .show()

+----------+-------------------+---------------+------+
|      date|new_cases_yesterday|new_cases_today| delta|
+----------+-------------------+---------------+------+
|2021-03-25|             8769.0|         9128.0| 359.0|
|2021-03-26|             9128.0|         9073.0| -55.0|
|2021-03-27|             9073.0|         8783.0|-290.0|
|2021-03-28|             8783.0|         8979.0| 196.0|
|2021-03-29|             8979.0|         8589.0|-390.0|
|2021-03-30|             8589.0|         8162.0|-427.0|
|2021-03-31|             8162.0|         8156.0|  -6.0|
+----------+-------------------+---------------+------+

